# Experiment 2.13: Activation Curvature Toy Sweep

## Pair identity

This notebook is the reporting and analysis counterpart to `run_experiment.py` in the same directory. It **imports the script module and uses its returned structured results** rather than duplicating the training logic, so the notebook and script remain behaviorally aligned.

## Truthful scope of this first-pass study

- single seed (`SEED = 42`)
- one synthetic dataset and one initialization recipe
- final losses only after 500 training steps
- 6 activations
- 54 total training runs (`6 × [SGD, Muon, static penalty, 6 alpha-sweep partial blends]`)
- current verdict targets **partial-blend recovery** as the primary analyzed metric
- the **static weight orthogonality penalty** remains a reported control, not the metric that currently drives the pass/fail verdict

This means the notebook can support, at most, a careful **toy-study statement** about how activation curvature relates to the recoverability of Muon's advantage by a partial orthogonal-gradient surrogate.

In [ ]:
import importlib.util
import platform
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)
np.set_printoptions(precision=4, suppress=True)

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

## Notebook-safe path resolution and script import

The notebook does **not** rely on `__file__`. Instead, it searches upward from the current working directory for the repository root that contains this experiment pair, then imports the script explicitly from disk.

In [ ]:
def find_repo_root(start: Path) -> Path:
    expected = Path("experiments/Tier_1_Core_Mechanism_Tests/ACTIVATION_CURVATURE_sweep/run_experiment.py")
    for candidate in [start, *start.parents]:
        if (candidate / expected).exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root containing "
        "experiments/Tier_1_Core_Mechanism_Tests/ACTIVATION_CURVATURE_sweep/run_experiment.py"
    )

REPO_ROOT = find_repo_root(Path.cwd())
EXPERIMENT_DIR = REPO_ROOT / "experiments" / "Tier_1_Core_Mechanism_Tests" / "ACTIVATION_CURVATURE_sweep"
SCRIPT_PATH = EXPERIMENT_DIR / "run_experiment.py"

spec = importlib.util.spec_from_file_location("activation_curvature_toy_sweep", SCRIPT_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f"Unable to create import spec for {SCRIPT_PATH}")
experiment = importlib.util.module_from_spec(spec)
spec.loader.exec_module(experiment)

print("Repository root:", REPO_ROOT)
print("Experiment directory:", EXPERIMENT_DIR)
print("Imported script:", SCRIPT_PATH)

## Reproducibility, runtime, and configuration

This cell executes the script's `run_experiment(verbose=False)` function exactly once and records both the script-reported runtime and the notebook wall-clock runtime.

In [ ]:
notebook_start = time.time()
results = experiment.run_experiment(verbose=False)
notebook_elapsed = time.time() - notebook_start

config = results["config"]
summary = results["summary"]

config_rows = [
    ("script_path", str(SCRIPT_PATH.relative_to(REPO_ROOT))),
    ("python_version", sys.version.split()[0]),
    ("platform", platform.platform()),
    ("numpy_version", np.__version__),
    ("scope", config["scope"]),
    ("seed", config["seed"]),
    ("architecture", f"depth={config['depth']}, width={config['width']}"),
    ("steps", config["num_steps"]),
    ("batch_size", config["batch_size"]),
    ("input_dim", config["input_dim"]),
    ("output_dim", config["output_dim"]),
    ("lr_sgd", config["lr_sgd"]),
    ("lr_muon", config["lr_muon"]),
    ("ortho_lambda", config["ortho_lambda"]),
    ("ns_iters", config["ns_iters"]),
    ("alpha_sweep", config["alpha_sweep"]),
    ("training_runs_per_activation", config["training_runs_per_activation"]),
    ("total_training_runs", config["total_training_runs"]),
    ("primary_analysis_metric", config["primary_analysis_metric"]),
    ("control_metric", config["control_metric"]),
    ("script_reported_runtime_sec", round(summary["runtime_sec"], 3)),
    ("notebook_wall_clock_sec", round(notebook_elapsed, 3)),
]
config_df = pd.DataFrame(config_rows, columns=["field", "value"])
display(config_df)

print("Overall verdict:", summary["overall_verdict"])
print("Overall message:", summary["overall_message"])
print("Partial-blend note:", config["partial_blend_note"])

## Structured per-activation results table

The table below is the notebook's main quantitative summary of the script output. `recovery_penalty` is the static weight-penalty control; `recovery_partial` is the currently tested surrogate metric used by the script's main trend tests.

In [ ]:
rows = []
for name in results["sorted_activation_names"]:
    r = results["per_activation"][name]
    rows.append(
        {
            "activation": name,
            "curvature": r["curvature"],
            "loss_sgd": r["loss_sgd"],
            "loss_muon": r["loss_muon"],
            "loss_penalty": r["loss_penalty"],
            "loss_best_partial": r["loss_partial"],
            "best_alpha": r["best_alpha"],
            "recovery_penalty": r["recovery_penalty"],
            "recovery_partial": r["recovery_partial"],
            "sgd_muon_gap": r["sgd_muon_gap"],
            "muon_beats_sgd": r["muon_beats_sgd"],
        }
    )
summary_df = pd.DataFrame(rows)
display(summary_df.round(4))

### Interpretation

- A positive recovery means the method improves on SGD toward Muon; a negative recovery means it is **worse than SGD**.
- If `recovery_penalty` is systematically weak or negative, the present pair should **not** be described as evidence that a static orthogonality penalty recovers Muon's benefit.
- `best_alpha` reports the strongest result within the current finite sweep only; it is not a claim of full optimizer equivalence.

## Final losses by activation

This figure compares the four reported endpoints for each activation:

1. SGD baseline
2. Muon
3. SGD + static weight orthogonality penalty
4. the **best** partial orthogonal-gradient blend found within the current alpha sweep

In [ ]:
plot_df = summary_df.copy()
labels = plot_df["activation"].tolist()
x = np.arange(len(labels))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 4.8))
ax.bar(x - 1.5 * width, plot_df["loss_sgd"], width, label="SGD")
ax.bar(x - 0.5 * width, plot_df["loss_muon"], width, label="Muon")
ax.bar(x + 0.5 * width, plot_df["loss_penalty"], width, label="Static penalty")
ax.bar(x + 1.5 * width, plot_df["loss_best_partial"], width, label="Best partial blend")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=25, ha="right")
ax.set_ylabel("Final loss")
ax.set_title("Final loss comparison by activation")
ax.legend(frameon=False, ncol=4)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

### Interpretation

This plot should be read conservatively. The only thing it directly shows is endpoint performance after a fixed 500-step budget on one synthetic task. In particular:

- Muon vs SGD quantifies whether there is any optimizer advantage to recover.
- The static-penalty bar is a control condition, not a supported mechanistic success story unless it actually closes the gap.
- The best-partial bar shows how much of Muon's advantage can be mimicked by mixing the raw gradient with an orthogonalized surrogate under the current learning-rate and scaling choices.

In [ ]:
penalty_improves = int((summary_df["loss_penalty"] < summary_df["loss_sgd"]).sum())
partial_improves = int((summary_df["loss_best_partial"] < summary_df["loss_sgd"]).sum())
muon_improves = int((summary_df["loss_muon"] < summary_df["loss_sgd"]).sum())

print(f"Muon beats SGD for {muon_improves}/{len(summary_df)} activations.")
print(f"Static penalty beats SGD for {penalty_improves}/{len(summary_df)} activations.")
print(f"Best partial blend beats SGD for {partial_improves}/{len(summary_df)} activations.")

## Curvature vs recovery

The next plot separates the two recovery notions that were previously conflated:

- **Static penalty recovery**: how much of Muon's advantage is recovered by SGD + weight orthogonality penalty
- **Partial-blend recovery**: how much is recovered by the best norm-matched orthogonal-gradient blend in the current alpha sweep

The script's current pass/fail trend tests use **partial-blend recovery**, not penalty recovery.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.axhline(0.0, color="black", linewidth=1.0, linestyle="--", alpha=0.6)
ax.scatter(summary_df["curvature"], summary_df["recovery_penalty"], s=80, label="Static penalty recovery")
ax.scatter(summary_df["curvature"], summary_df["recovery_partial"], s=80, marker="s", label="Best partial-blend recovery")

for _, row in summary_df.iterrows():
    ax.annotate(row["activation"], (row["curvature"], row["recovery_partial"]), xytext=(5, 5), textcoords="offset points", fontsize=9)

ax.set_xlabel("Curvature proxy: mean |f''(x)| over x ~ N(0, 1)")
ax.set_ylabel("Recovery of Muon's SGD advantage (%)")
ax.set_title("Curvature vs recovery: control metric vs primary analyzed metric")
ax.legend(frameon=False)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

### Interpretation

This figure is the cleanest way to avoid overclaiming:

- If the static-penalty series stays near or below zero, then the current pair does **not** support a claim that a simple weight orthogonality penalty recovers Muon's edge.
- If the partial-blend series shows a negative trend with curvature, the current code supports only a **surrogate trend statement**: recovery by the partial orthogonalized-step blend becomes weaker for higher-curvature activations in this toy setup.
- Because this is a single-seed final-loss study, any apparent trend should be treated as provisional rather than as a stable effect-size estimate.

In [ ]:
print(f"Spearman rho (curvature vs penalty recovery): {summary['spearman_penalty']:.4f}")
print(f"Spearman rho (curvature vs partial recovery): {summary['spearman_partial']:.4f}")
print(
    "Pairwise concordance for partial recovery: "
    f"{results['concordance']['n_concordant']}/{results['concordance']['n_pairs']} "
    f"({summary['concordance_partial']:.0%})"
)

## Best alpha by activation

The current partial-blend experiment searches over a finite alpha grid. The best alpha summarizes which amount of orthogonalized-step mixing worked best for each activation **under the current toy settings**.

Crucially, this should not be described as interpolation all the way to Muon, because the partial-blend routine uses `LR_SGD` and a norm-matched orthogonalized step, whereas Muon uses `LR_MUON` and an unscaled orthogonalized update.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.2))
ax.bar(summary_df["activation"], summary_df["best_alpha"], color="tab:purple")
ax.set_ylim(0, 1.0)
ax.set_ylabel("Best alpha in current sweep")
ax.set_title("Best partial-blend alpha by activation")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=25, ha="right")
fig.tight_layout()
plt.show()

### Interpretation

The best-alpha summary is descriptive, not a standalone hypothesis test. It can be useful for seeing whether higher-curvature activations prefer more or less orthogonalized blending within this narrow setup, but it does not by itself establish optimizer equivalence, causal mechanism, or robustness.

In [ ]:
alpha_rows = []
for name in results["sorted_activation_names"]:
    for record in results["per_activation"][name]["alpha_sweep"]:
        alpha_rows.append(
            {
                "activation": name,
                "alpha": record["alpha"],
                "loss": record["loss"],
            }
        )
alpha_df = pd.DataFrame(alpha_rows)
alpha_pivot = alpha_df.pivot(index="activation", columns="alpha", values="loss")
display(alpha_pivot.round(4))

## Current hypothesis tests and verdict

The table below reproduces the script's current thresholded tests. These tests are still toy-study diagnostics; they are not substitutes for uncertainty quantification across seeds.

In [ ]:
tests_df = pd.DataFrame(results["tests"])
display(tests_df[["id", "description", "analysis_target", "metric_display", "passed"]])

print("Overall verdict:", summary["overall_verdict"])
print("Overall message:", summary["overall_message"])

## Calibrated conclusion

This notebook intentionally ends with a narrow conclusion that matches the implemented metrics rather than the strongest possible narrative.

In [ ]:
zero_group = summary["group_summary"]["zero_curvature_activation_names"]
nonzero_group = summary["group_summary"]["nonzero_curvature_activation_names"]
mean_zero = summary["group_summary"]["mean_zero_curvature_recovery_partial"]
mean_nonzero = summary["group_summary"]["mean_nonzero_curvature_recovery_partial"]

print("1. This pair currently implements a single-seed, final-loss-only toy sweep.")
print(
    "2. The script and notebook now agree on the primary analyzed metric: "
    "partial-blend recovery, with static penalty recovery shown as a control."
)
print(
    f"3. In this run, Muon beats SGD for {summary['muon_wins']}/{summary['activation_count']} activations, "
    f"with Spearman rho={summary['spearman_partial']:.4f} for curvature vs partial recovery."
)
print(
    f"4. Mean static-penalty recovery is {summary['mean_recovery_penalty']:.1f}%, whereas mean partial-blend recovery is "
    f"{summary['mean_recovery_partial']:.1f}%."
)
if mean_zero is not None and mean_nonzero is not None:
    print(
        f"5. Zero-curvature activations ({zero_group}) have mean partial recovery {mean_zero:.1f}%, "
        f"compared with {mean_nonzero:.1f}% for nonzero-curvature activations ({nonzero_group})."
    )
print(
    "6. Therefore, the present run can support only a cautious claim about a partial orthogonal-gradient surrogate trend "
    "in this toy setting. It does not, by itself, justify a claim that a static weight orthogonality penalty reliably "
    "recovers Muon's advantage."
)
print(
    "7. The next serious verification step is a small multi-seed extension and, ideally, trajectory recording rather than "
    "relying exclusively on one final-loss snapshot per condition."
)